In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/minhng-178/Locality-iN-Locality.git

In [ ]:
cd /content/Locality-iN-Locality

In [ ]:
!pip install torchattacks timm einops ptflops
!pip install git+https://github.com/sovrasov/flops-counter.pytorch.git

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import WeightedRandomSampler

import torchvision.utils
from torchvision import models
import torchvision.datasets as dsets
import torchvision.transforms as transforms

import torchattacks
from torchattacks import PGD, FGSM

import shutil

import time
from ptflops import get_model_complexity_info

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
print('PyTorch', torch.__version__)
print('Torchvision', torchvision.__version__)
print('Torchattacks', torchattacks.__version__)
print('Numpy', np.__version__)

## GTSRB (German Traffic Sign Recognition Benchmark)
Dataset uses .ppm images, 43 classes, ~39k training / ~12k test images

In [ ]:
# !mkdir data

# !curl --url https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip -o data/GTSRB_Final_Training_Images.zip
# !curl --url https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_Images.zip -o data/GTSRB_Final_Test_Images.zip
# !curl --url https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_GT.zip -o data/GTSRB_Final_Test_GT.zip

In [ ]:
# !unzip data/GTSRB_Final_Training_Images.zip -d data/ > /dev/null 2>&1
# !unzip data/GTSRB_Final_Test_Images.zip -d data/ > /dev/null 2>&1
# !unzip data/GTSRB_Final_Test_GT.zip -d data/

In [ ]:
# Organize test images into class folders for ImageFolder
data_dir = './data/GTSRB'
images_dir = os.path.join(data_dir, 'Final_Test/Images')

test_dir = os.path.join(data_dir, 'test')
os.makedirs(test_dir, exist_ok=True)

with open('./data/GT-final_test.csv') as f:
  image_names = f.readlines()

for text in image_names[1:]:
  classes = int(text.split(';')[-1])
  image_name = text.split(';')[0]

  test_class_dir = os.path.join(test_dir, f"{classes:04d}")
  os.makedirs(test_class_dir, exist_ok=True)
  image_path = os.path.join(images_dir, image_name)

  shutil.copy(image_path, test_class_dir)

In [ ]:
# GTSRB-specific normalization (computed from training set)
# IMPORTANT: Model architecture expects mean/std = (0.5, 0.5, 0.5) for pretrained weights.
# Since we train from scratch, we use GTSRB actual stats for faster convergence.
GTSRB_MEAN = [0.3337, 0.3064, 0.3171]
GTSRB_STD  = [0.2672, 0.2564, 0.2629]

# NOTE: KHÔNG dùng RandomHorizontalFlip - lật ngang biển báo sẽ tạo class sai
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.5),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=GTSRB_MEAN, std=GTSRB_STD),
    transforms.RandomErasing(p=0.25),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=GTSRB_MEAN, std=GTSRB_STD),
])

In [ ]:
batch_size = 64

trainset = torchvision.datasets.ImageFolder(
    root='./data/GTSRB/Final_Training/Images',
    transform=train_transform
)

testset = torchvision.datasets.ImageFolder(
    root='./data/GTSRB/test',
    transform=test_transform
)

# Class-balanced sampling (GTSRB bị imbalanced nặng)
train_root = './data/GTSRB/Final_Training/Images'
class_list = sorted(os.listdir(train_root))
class_counts = [len(os.listdir(os.path.join(train_root, c))) for c in class_list]
weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
sample_weights = weights[trainset.targets]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = torch.utils.data.DataLoader(
    dataset=trainset,
    batch_size=batch_size,
    sampler=sampler
)

test_loader = torch.utils.data.DataLoader(
    dataset=testset,
    batch_size=batch_size,
    shuffle=False
)

print(f'Train classes: {len(trainset.classes)}, samples: {len(trainset)}')
print(f'Test classes: {len(testset.classes)}, samples: {len(testset)}')
print(f'Train batches: {len(train_loader)}, Test batches: {len(test_loader)}')

In [ ]:
batch = next(iter(train_loader))
train_data = batch[0]

In [ ]:
def denormalize_image(image, mean=GTSRB_MEAN, std=GTSRB_STD):
    image = image.clone()
    for t, m, s in zip(image, mean, std):
        t.mul_(s).add_(m)
    return image

def normalize_image(image):
    image_min = image.min()
    image_max = image.max()
    image = torch.clamp(image, image_min, image_max)
    image = (image - image_min) / (image_max - image_min + 1e-5)
    return image

def plot_images(images, labels, classes, normalize=True):
    n_images = len(images)
    rows = int(np.sqrt(n_images))
    cols = int(np.sqrt(n_images))
    fig = plt.figure(figsize=(20, 20))
    for i in range(rows*cols):
        ax = fig.add_subplot(rows, cols, i+1)
        image = images[i]
        if normalize:
            image = normalize_image(image)
        ax.imshow(image.permute(1, 2, 0).cpu().numpy())
        ax.set_title(classes[labels[i]])
        ax.axis('off')

In [ ]:
classes = trainset.classes
plot_images(batch[0], batch[1], classes)

## LNL Model (Locality-iN-Locality)
Using LNL_Ti (6M params) - tốt hơn cho dataset nhỏ, ít overfit hơn LNL_S (24M)

In [ ]:
from LNL import LNL_Ti as small

# Create model with drop_path_rate for regularization
model = small(pretrained=False, num_classes=43, drop_path_rate=0.1)
model = model.to(device)

In [ ]:
model.head

## Train Locality-iN-Locality
Improved training: AdamW + Cosine Warmup + Label Smoothing + Mixup

In [ ]:
# Mixup implementation
def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
num_epochs = 25

# Label smoothing
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# AdamW - hội tụ nhanh hơn nhiều cho Transformers (tăng LR bù cho ít epochs)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)

# Cosine annealing with linear warmup
warmup_epochs = 3
warmup_scheduler = LinearLR(optimizer, start_factor=0.01, total_iters=warmup_epochs)
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs - warmup_epochs, eta_min=1e-6)
scheduler = SequentialLR(optimizer, [warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])

In [ ]:
scaler = torch.amp.GradScaler() if device.type == 'cuda' else None

for epoch in range(num_epochs):
    model.train()
    total_batch = len(train_loader)
    running_loss = 0.0

    for i, (batch_images, batch_labels) in enumerate(train_loader):
        X = batch_images.to(device)
        Y = batch_labels.to(device)

        # Mixup
        if np.random.random() < 0.5:
            X, Y_a, Y_b, lam = mixup_data(X, Y, alpha=0.2)
        else:
            Y_a, Y_b, lam = Y, Y, 1.0

        optimizer.zero_grad()

        if scaler is not None:
            with torch.amp.autocast('cuda'):
                pre = model(X)
                cost = mixup_criterion(loss_fn, pre, Y_a, Y_b, lam)
            scaler.scale(cost).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            pre = model(X)
            cost = mixup_criterion(loss_fn, pre, Y_a, Y_b, lam)
            cost.backward()
            optimizer.step()

        running_loss += cost.item()

        if (i+1) % 100 == 0:
            avg_loss = running_loss / (i+1)
            print('Epoch [%d/%d], Iter [%d/%d], Loss: %.6f, LR: %.2e'
                 % (epoch+1, num_epochs, i+1, total_batch, avg_loss,
                    optimizer.param_groups[0]['lr']))

    scheduler.step()
    print(f'=== Epoch {epoch+1}/{num_epochs} complete, Avg Loss: {running_loss/total_batch:.6f} ===')

## Test

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Standard accuracy: %.2f %%' % (100 * float(correct) / total))

## FGSM attack

In [ ]:
model.eval()

correct = 0
total = 0

atk = FGSM(model, eps=0.01)

for images, labels in test_loader:
    images = atk(images, labels).to(device)
    outputs = model(images)

    _, predicted = torch.max(outputs.data, 1)

    total += labels.size(0)
    correct += (predicted == labels.to(device)).sum().item()

print('Robust accuracy (FGSM): %.2f %%' % (100 * float(correct) / total))

## PGD attack

In [ ]:
model.eval()

correct = 0
total = 0

atk = PGD(model, eps=0.01, alpha=2/255, steps=5, random_start=False)

for images, labels in test_loader:
    images = atk(images, labels).to(device)
    outputs = model(images)

    _, predicted = torch.max(outputs.data, 1)

    total += labels.size(0)
    correct += (predicted == labels.to(device)).sum().item()

print('Robust accuracy (PGD): %.2f %%' % (100 * float(correct) / total))

## Train LNL-MoEx (Moment Exchange variant)

In [ ]:
from LNL_MoEx import LNL_MoEx_Ti as small_moex
model_moex = small_moex(pretrained=False, num_classes=43, drop_path_rate=0.1)
model_moex = model_moex.to(device)

In [ ]:
num_epochs_moex = 20
moex_lam = .9
moex_prob = .7

In [ ]:
loss_fn_moex = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_moex = torch.optim.AdamW(model_moex.parameters(), lr=1e-3, weight_decay=0.05)

warmup_scheduler_moex = LinearLR(optimizer_moex, start_factor=0.01, total_iters=3)
cosine_scheduler_moex = CosineAnnealingLR(optimizer_moex, T_max=num_epochs_moex - 3, eta_min=1e-6)
scheduler_moex = SequentialLR(optimizer_moex, [warmup_scheduler_moex, cosine_scheduler_moex], milestones=[3])

In [ ]:
scaler_moex = torch.amp.GradScaler() if device.type == 'cuda' else None

for epoch in range(num_epochs_moex):
    model_moex.train()
    total_batch = len(train_loader)
    running_loss = 0.0

    for i, (input, target) in enumerate(train_loader):
        input = input.to(device)
        target = target.to(device)

        prob = torch.rand(1).item()
        if prob < moex_prob:
            swap_index = torch.randperm(input.size(0), device=input.device)
            with torch.no_grad():
                target_a = target
                target_b = target[swap_index]

            optimizer_moex.zero_grad()
            if scaler_moex is not None:
                with torch.amp.autocast('cuda'):
                    output = model_moex(input, swap_index=swap_index, moex_norm='pono',
                                        moex_epsilon=1e-5, moex_layer='stem',
                                        moex_positive_only=False)
                    cost = loss_fn_moex(output, target_a) * moex_lam + \
                           loss_fn_moex(output, target_b) * (1. - moex_lam)
                scaler_moex.scale(cost).backward()
                scaler_moex.step(optimizer_moex)
                scaler_moex.update()
            else:
                output = model_moex(input, swap_index=swap_index, moex_norm='pono',
                                    moex_epsilon=1e-5, moex_layer='stem',
                                    moex_positive_only=False)
                cost = loss_fn_moex(output, target_a) * moex_lam + \
                       loss_fn_moex(output, target_b) * (1. - moex_lam)
                cost.backward()
                optimizer_moex.step()
        else:
            optimizer_moex.zero_grad()
            if scaler_moex is not None:
                with torch.amp.autocast('cuda'):
                    output = model_moex(input)
                    cost = loss_fn_moex(output, target)
                scaler_moex.scale(cost).backward()
                scaler_moex.step(optimizer_moex)
                scaler_moex.update()
            else:
                output = model_moex(input)
                cost = loss_fn_moex(output, target)
                cost.backward()
                optimizer_moex.step()

        running_loss += cost.item()

        if (i+1) % 100 == 0:
            avg_loss = running_loss / (i+1)
            print('Epoch [%d/%d], Iter [%d/%d], Loss: %.6f, LR: %.2e'
                 % (epoch+1, num_epochs_moex, i+1, total_batch, avg_loss,
                    optimizer_moex.param_groups[0]['lr']))

    scheduler_moex.step()
    print(f'=== MoEx Epoch {epoch+1}/{num_epochs_moex} complete, Avg Loss: {running_loss/total_batch:.6f} ===')

In [ ]:
model_moex.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model_moex(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Standard accuracy after MoEx training: %.2f %%' % (100 * float(correct) / total))

## Number of Parameters

In [ ]:
macs, params = get_model_complexity_info(model, (3, 224, 224), as_strings=True,
                                         print_per_layer_stat=False, verbose=False)
print('{:<30}  {:<8}'.format('Computational complexity: ', macs))
print('{:<30}  {:<8}'.format('Number of parameters: ', params))

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('{:<30}  {:<8}'.format('Total params: ', f'{total_params:,}'))
print('{:<30}  {:<8}'.format('Trainable params: ', f'{trainable_params:,}'))

## Save Model

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': num_epochs,
}, 'LNL_Ti_GTSRB.pth')
print('Model saved to LNL_Ti_GTSRB.pth')